# Debug Hybrid Scaling Issue

**Problem:** Perfect k-space → terrible focal plane

**Hypothesis:** There's a scaling mismatch between:
- Analytical jinc²(k_ρ × R) sampling
- Fraunhofer propagation: x = (λf/2π) × kx

In [1]:
import sys
sys.path.insert(0, '..')

import numpy as np
import matplotlib.pyplot as plt
from scipy.special import j1

from monte_carlo.angular_spectrum_hybrid import AngularSpectrumHybrid
from monte_carlo import metrics

In [2]:
wavelength = 0.6328  # microns
focal_length = 10.0  # mm
NA = 0.1
n_medium = 1.0
n_photons = 100000

sim = AngularSpectrumHybrid(
    n_photons=n_photons,
    wavelength=wavelength,
    focal_length=focal_length,
    numerical_aperture=NA,
    n_medium=n_medium,
    random_seed=42
)

results = sim.propagate()
x_focal, y_focal, _ = results['focal_positions']
kx, ky, kz = results['k_vectors']

## Check Scale Factors and Units

In [3]:
print("PARAMETERS:")
print(f"  λ = {wavelength} μm")
print(f"  f = {focal_length} mm = {focal_length * 1000} μm")
print(f"  NA = {NA}")
print(f"\nCALCULATED VALUES:")
print(f"  k = 2π/λ = {sim.k:.4f} rad/μm")
print(f"  k_max = k × NA = {sim.k_max:.4f} rad/μm")
print(f"  Aperture radius R = {sim.aperture_radius:.4f} mm = {sim.aperture_radius * 1000:.1f} μm")
print(f"\nSCALE FACTORS:")
fraunhofer_scale = (wavelength * focal_length) / (2 * np.pi)
print(f"  Fraunhofer: (λf/2π) = {fraunhofer_scale:.6f} μm·mm/rad")
print(f"\nEXPECTED:")
airy_radius = 1.22 * wavelength / NA
print(f"  Airy radius = 1.22λ/NA = {airy_radius:.4f} μm")

PARAMETERS:
  λ = 0.6328 μm
  f = 10.0 mm = 10000.0 μm
  NA = 0.1

CALCULATED VALUES:
  k = 2π/λ = 9.9292 rad/μm
  k_max = k × NA = 0.9929 rad/μm
  Aperture radius R = 1.0000 mm = 1000.0 μm

SCALE FACTORS:
  Fraunhofer: (λf/2π) = 1.007132 μm·mm/rad

EXPECTED:
  Airy radius = 1.22λ/NA = 7.7202 μm


## Check Jinc Argument Range

In [4]:
k_rho = np.sqrt(kx**2 + ky**2)

print("\nJINC FUNCTION ARGUMENTS:")
print(f"  k_ρ range: [{np.min(k_rho):.4f}, {np.max(k_rho):.4f}] rad/μm")
print(f"  R = {sim.aperture_radius:.4f} mm = {sim.aperture_radius * 1000:.1f} μm")

# BUG CHECK: Are we mixing units?
arg_mm = k_rho * sim.aperture_radius  # k_rho in rad/μm, R in mm
arg_um = k_rho * (sim.aperture_radius * 1000)  # k_rho in rad/μm, R in μm

print(f"\n  If R in mm: k_ρ × R range = [{np.min(arg_mm):.1f}, {np.max(arg_mm):.1f}]")
print(f"  If R in μm: k_ρ × R range = [{np.min(arg_um):.1f}, {np.max(arg_um):.1f}]")
print(f"\n  First zero of jinc is at x ≈ 3.83")
print(f"  If using R in μm, jinc²({np.max(arg_um):.1f}) ≈ 0 for most photons!")
print(f"\n  ⚠ UNIT MISMATCH! k_ρ is in rad/μm but R is in mm!")


JINC FUNCTION ARGUMENTS:
  k_ρ range: [0.0022, 0.9929] rad/μm
  R = 1.0000 mm = 1000.0 μm

  If R in mm: k_ρ × R range = [0.0, 1.0]
  If R in μm: k_ρ × R range = [2.2, 992.9]

  First zero of jinc is at x ≈ 3.83
  If using R in μm, jinc²(992.9) ≈ 0 for most photons!

  ⚠ UNIT MISMATCH! k_ρ is in rad/μm but R is in mm!


## Check Actual Focal Plane Distribution

In [5]:
r_focal = np.sqrt(x_focal**2 + y_focal**2)

print("\nFOCAL PLANE PHOTON DISTRIBUTION:")
print(f"  Mean radius: {np.mean(r_focal):.4f} μm")
print(f"  Max radius:  {np.max(r_focal):.4f} μm")
print(f"  95th percentile: {np.percentile(r_focal, 95):.4f} μm")
print(f"\n  Expected Airy radius: {airy_radius:.4f} μm")
print(f"  Actual max radius is {np.max(r_focal)/airy_radius:.2f}x Airy radius")

if np.max(r_focal) < airy_radius * 0.5:
    print("\n  ⚠ Photons are too concentrated! Distribution is too narrow.")


FOCAL PLANE PHOTON DISTRIBUTION:
  Mean radius: 0.6499 μm
  Max radius:  1.0000 μm
  95th percentile: 0.9711 μm

  Expected Airy radius: 7.7202 μm
  Actual max radius is 0.13x Airy radius

  ⚠ Photons are too concentrated! Distribution is too narrow.


## Diagnose: What's the Correct Aperture Radius?

In [6]:
print("\n" + "="*70)
print("DIAGNOSIS")
print("="*70)
print("\nThe jinc function argument should be k_ρ × R where:")
print("  - k_ρ is the transverse wave vector (rad/μm)")
print("  - R is the aperture radius")
print("\nFor proper scaling:")
print("  - If k_ρ is in rad/μm, then R must be in μm")
print("  - Otherwise k_ρ × R will have wrong units!")
print("\nCURRENT CODE:")
print(f"  k_ρ = {np.max(k_rho):.4f} rad/μm")
print(f"  R = {sim.aperture_radius:.4f} mm (WRONG UNITS!)")
print(f"  k_ρ × R = {np.max(k_rho) * sim.aperture_radius:.4f} rad·mm/μm")
print("\nSHOULD BE:")
print(f"  k_ρ = {np.max(k_rho):.4f} rad/μm")
print(f"  R = {sim.aperture_radius * 1000:.1f} μm")
print(f"  k_ρ × R = {np.max(k_rho) * sim.aperture_radius * 1000:.1f} rad")
print("\nBut that makes jinc({np.max(k_rho) * sim.aperture_radius * 1000:.1f}) ≈ 0!")
print("\n→ There's a fundamental issue with the scaling!")
print("="*70)


DIAGNOSIS

The jinc function argument should be k_ρ × R where:
  - k_ρ is the transverse wave vector (rad/μm)
  - R is the aperture radius

For proper scaling:
  - If k_ρ is in rad/μm, then R must be in μm
  - Otherwise k_ρ × R will have wrong units!

CURRENT CODE:
  k_ρ = 0.9929 rad/μm
  R = 1.0000 mm (WRONG UNITS!)
  k_ρ × R = 0.9929 rad·mm/μm

SHOULD BE:
  k_ρ = 0.9929 rad/μm
  R = 1000.0 μm
  k_ρ × R = 992.9 rad

But that makes jinc({np.max(k_rho) * sim.aperture_radius * 1000:.1f}) ≈ 0!

→ There's a fundamental issue with the scaling!
